# Эксперимент 5: Демонстрация полного пайплайна (Pipeline Demo)

Как `ABTestRunner` автоматически выбирает правильный метод для каждого типа данных через `DecisionEngine`.

Четыре сценария:
- **E-commerce** (rho=0.46) - CUPED
- **Low Correlation** (rho=0.17) - baseline t-test
- **Ratio Metric** - Delta Method
- **Segmented** (высокая between-group var) - Post-Stratification

In [15]:
import pandas as pd
import numpy as np
import sys, os

sys.path.append(os.path.abspath('..'))

from ab_framework.data.generator import (
    EcommerceScenario, LowCorrelationScenario, RatioScenario, SegmentedScenario
)
from ab_framework.pipeline.runner import ABTestRunner

pd.set_option('display.max_colwidth', None)
N_USERS  = 10_000
EFFECT   = 0.05   
SEED     = 42


## Сценарий 1: E-commerce (rho=0.46)

Денежная метрика с pre-периодом. Высокая корреляция pre/post - CUPED.

In [ ]:
df_ecomm = EcommerceScenario().generate(    
    n_users=N_USERS, effect_size=EFFECT, seed=SEED
)
config_ecomm = {
    'target_metric':          'revenue',
    'group_column':           'group',
    'pre_experiment_metric':  'pre_revenue',
    'categorical_covariates': ['device_type', 'age_group'],
    'metric_type':            'continuous',
}

print('Данные:', df_ecomm.shape)
print('Корреляция pre/post:', round(df_ecomm['pre_revenue'].corr(df_ecomm['revenue']), 3))
df_ecomm.head(3)


Данные: (10000, 6)
Корреляция pre/post: 0.459


,user_id,group,pre_revenue,revenue,device_type,age_group
0,1,1,514.80,547.23,iOS,35-44
1,2,1,175.57,1794.29,Android,25-34
2,3,0,735.36,218.51,iOS,18-24


In [17]:
r1 = ABTestRunner(df_ecomm, config_ecomm).run()

print(' Решение DecisionEngine')
print(r1['execution_plan'])
print()
print('Результаты теста')
print(f"Метод:              {r1['method_used']}")
print(f"p-value:            {r1['p_value']:.4f}")
print(f"Значимо (α=0.05):   {r1['is_significant']}")
print(f"Оценка эффекта:     {r1['effect_estimate']:.2f}")
print(f"Снижение дисперсии: {r1['variance_reduction_pct']:.1f}%")


 Решение DecisionEngine
Обоснование алгоритма:
- Винсоризация (предобработка выбросов): эксцесс = 21.7 > 10. Применяется ДО variance reduction, не как самостоятельный метод.
- CUPED: корреляция pre/post rho=0.459 >= 0.25. Ожидаемое снижение дисперсии: 21.1% (теор. ускорение = 1.27x).
- Welch t-test для оценки разницы средних.

Результаты теста
Метод:              standard_cuped + welch t-test
p-value:            0.0323
Значимо (α=0.05):   True
Оценка эффекта:     19.59
Снижение дисперсии: 36.2%


## Сценарий 2: Low Correlation (rho=0.17)

Аналогичные данные, но слабая корреляция pre/post. CUPED не даёт прироста - движок выбирает baseline t-test.

In [18]:
df_lowcorr = LowCorrelationScenario().generate(  
    n_users=N_USERS, effect_size=EFFECT, seed=SEED
)
config_lowcorr = {
    'target_metric':          'revenue',
    'group_column':           'group',
    'pre_experiment_metric':  'pre_revenue',
    'categorical_covariates': ['device_type', 'age_group'],
    'metric_type':            'continuous',
}

print('Корреляция pre/post:', round(df_lowcorr['pre_revenue'].corr(df_lowcorr['revenue']), 3))

r2 = ABTestRunner(df_lowcorr, config_lowcorr).run()
print()
print('Решение DecisionEngine ')
print(r2['execution_plan'])
print()
print(f"Метод: {r2['method_used']}")
print(f"p-value: {r2['p_value']:.4f}  |  Значимо: {r2['is_significant']}")
print(f"Снижение дисперсии: {r2['variance_reduction_pct']:.1f}%  (ожидаем ~0% от VR; реальное снижение — от предобработки Винсоризацией)")


Корреляция pre/post: 0.173

Решение DecisionEngine 
Обоснование алгоритма:
- Винсоризация (предобработка выбросов): эксцесс = 17.4 > 10. Применяется ДО variance reduction, не как самостоятельный метод.
- Variance reduction не применяется: rho=0.173 < 0.25 и/или between-group variance ниже порога 10%.
- Welch t-test для оценки разницы средних.

Метод: none + welch t-test
p-value: 0.0065  |  Значимо: True
Снижение дисперсии: 14.9%  (ожидаем ~0% от VR; реальное снижение — от предобработки Винсоризацией)


## Сценарий 3: Ratio Metric (ARPU = revenue / sessions)

Ratio-метрика требует Delta Method. Наивный t-test некорректен: не учитывает дисперсию знаменателя.

In [19]:
df_ratio = RatioScenario().generate(
    n_users=N_USERS, effect_size=EFFECT, seed=SEED
)
config_ratio = {
    'target_metric':     'revenue',
    'group_column':      'group',
    'denominator_metric':'n_sessions',
    'categorical_covariates': ['device_type'],
    'metric_type':       'ratio',
}

print('Колонки:', list(df_ratio.columns))
print(f"Средний ARPU (control):   "
      f"{df_ratio[df_ratio['group']==0]['revenue'].mean() / df_ratio[df_ratio['group']==0]['n_sessions'].mean():.2f}")
print(f"Средний ARPU (treatment): "
      f"{df_ratio[df_ratio['group']==1]['revenue'].mean() / df_ratio[df_ratio['group']==1]['n_sessions'].mean():.2f}")

r3 = ABTestRunner(df_ratio, config_ratio).run()
print()
print('Решение DecisionEngine')
print(r3['execution_plan'])
print()
print(f"Метод: {r3['method_used']}")
print(f"p-value: {r3['p_value']:.4f}  |  Значимо: {r3['is_significant']}")
print(f"ARPU control:   {r3['control_ratio']:.4f}")
print(f"ARPU treatment: {r3['treatment_ratio']:.4f}")


Колонки: ['user_id', 'group', 'revenue', 'n_sessions', 'device_type']
Средний ARPU (control):   13.87
Средний ARPU (treatment): 14.72

Решение DecisionEngine
Обоснование алгоритма:
- Винсоризация (предобработка выбросов): эксцесс = 28.4 > 10. Применяется ДО variance reduction, не как самостоятельный метод.
- Variance reduction не применяется: rho=0.000 < 0.25 и/или between-group variance ниже порога 10%.
- Delta Method: ratio-метрика. По бенчмарку: Power 78.0% vs 46.9% у baseline (+31.1%).

Метод: delta method
p-value: 0.0000  |  Значимо: True
ARPU control:   13.6519
ARPU treatment: 14.4712


## Сценарий 4: Segmented (высокая between-group variance)

Три сегмента с сильно разной дисперсией дохода. Нет pre-периода - движок выбирает Post-Stratification.

In [20]:
df_seg = SegmentedScenario().generate(
    n_users=N_USERS, effect_size=EFFECT, seed=SEED
)
config_seg = {
    'target_metric':          'revenue',
    'group_column':           'group',
    'categorical_covariates': ['device_type'],
    'metric_type':            'continuous',
}

seg_stats = df_seg.groupby('device_type')['revenue'].agg(['mean','std','count'])
print('Статистики по сегментам:')
print(seg_stats.to_string())

r4 = ABTestRunner(df_seg, config_seg).run()
print()
print('Решение DecisionEngine')
print(r4['execution_plan'])
print()
print(f"Метод: {r4['method_used']}")
print(f"p-value: {r4['p_value']:.4f}  |  Значимо: {r4['is_significant']}")
print(f"Снижение дисперсии: {r4['variance_reduction_pct']:.1f}%")


Статистики по сегментам:
                    mean         std  count
device_type                                
Android       521.484064  804.014695   5003
Web           177.295251  375.067783   1954
iOS          1260.033805  689.586016   3043

Решение DecisionEngine
Обоснование алгоритма:
- Винсоризация (предобработка выбросов): эксцесс = 18.1 > 10. Применяется ДО variance reduction, не как самостоятельный метод.
- Пост-стратификация: между-групповая дисперсия по 'device_type' = 24.9% от total variance (порог > 10%).
- Welch t-test для оценки разницы средних.

Метод: post_stratification + t-test
p-value: 0.1504  |  Значимо: False
Снижение дисперсии: 18.8%


---
> **Примечание по Segmented сценарию:** p-value = 0.1504 (незначимо) — это **ожидаемый результат**.
> Monte Carlo бенчмарк (1000 симуляций) показывает Power = 33.4% для Post-Stratification
> при n=10 000 и effect_size=5%. То есть в ~66% одиночных прогонов тест не обнаруживает
> эффект. Для достижения Power = 80% при данном сценарии требуется n ≈ 30 000 пользователей.
> Данный прогон (seed=42) попал в "неудачную" треть.

## Итоговая сводка

In [14]:
summary_data = [
    ('E-commerce (rho≈0.46)',  r1['method_used'], r1['p_value'], r1['is_significant'], r1['variance_reduction_pct']),
    ('Low Correlation',       r2['method_used'], r2['p_value'], r2['is_significant'], r2['variance_reduction_pct']),
    ('Ratio Metric',          r3['method_used'], r3['p_value'], r3['is_significant'], r3['variance_reduction_pct']),
    ('Segmented',             r4['method_used'], r4['p_value'], r4['is_significant'], r4['variance_reduction_pct']),
]

df_summary = pd.DataFrame(summary_data,
    columns=['Сценарий', 'Метод', 'p-value', 'Значимо', 'Var.Reduction%'])
df_summary['p-value'] = df_summary['p-value'].round(4)
df_summary['Var.Reduction%'] = df_summary['Var.Reduction%'].round(1)
print(df_summary.to_string(index=False))


             Сценарий                         Метод  p-value  Значимо  Var.Reduction%
E-commerce (rho≈0.46) standard_cuped + welch t-test   0.0323     True            36.2
      Low Correlation           none + welch t-test   0.0065     True            14.9
         Ratio Metric                  delta method   0.0000     True            22.0
            Segmented  post_stratification + t-test   0.1504    False            18.8
